# Master datasets for Theme 1 (SETAC Europe)

**What this notebook builds**
- **Theme 1 master tables** (membership health, retention, life cycle): renewal opportunities, year-end membership counts, and a person-level cohort table.

- most files have *many rows per person* (one person has several subscriptions, webinar registrations, newsletter activities...). If we merge those files together row by row, the rows multiply and every count becomes wrong. So we **summarise each file to ONE ROW PER PERSON first, and only then merge** on Contact ID.

## Step 1: Set Up

In [1]:
from pathlib import Path                     # Path: handles file and folder locations
import re                                    # re: text pattern tools (used to tidy file names)
import warnings                              
import pandas as pd                          
import numpy as np                           

warnings.filterwarnings("ignore")            # hide harmless warnings
pd.set_option("display.max_columns", 40)     # show more columns when printing tables
pd.set_option("display.width", 200)

current = Path.cwd().resolve()               # the folder this notebook is running from
project_root = next((p for p in [current, *current.parents] if (p / ".git").exists()), None)   # walk up to the project root (the folder containing .git)
if project_root is None:
    raise RuntimeError("Open the cloned SETAC-Team-12 project in VS Code and run this notebook from inside it.")
data_dir = project_root / "Fourth Rev - Data Edited"      # the agreed data folder
cleaned_dir = data_dir / "Cleaned"           # where outputs are saved
cleaned_dir.mkdir(exist_ok=True)

# ---- agreed rules (change only by team decision) ----
CUTOFF = pd.Timestamp("2026-09-30")          # the last date covered by the data
GRACE_DAYS = 31                              # SETAC: a membership stays 'Lapsed' for 1 month after expiry; after that the person must reapply
TRANSFER_DAYS = 180                          # a new subscription in another SETAC region within 180 days means "moved", not "lost"
LAST_RENEWABLE = pd.Timestamp("2026-08-31")  # Explorer and 3-year memberships could not be renewed after this date
MIGRATION_END = pd.Timestamp("2023-07-01")   # records created before this come from the old system (June 2023 migration)
ENG_START = CUTOFF - pd.Timedelta(days=365)  # start of the 12-month engagement window (Oct 2025 to Sep 2026)
print("Data folder:", data_dir)

Data folder: /Users/Bosola/Downloads/SETAC-Team-12/Fourth Rev - Data Edited


## Step 2: Load only the columns we need
`usecols` loads only the columns we list, so every other column is **dropped**. The comments say why each kept column matters. Columns not listed are dropped for one of these reasons: completely or mostly empty, duplicates of another column, personal details we do not need (e.g. pronouns)

### 2.1 Subscriptions Europe (the backbone of Theme 1)

In [2]:
SUBS_KEEP = [
    "Contact - ID",              # KEY: links the subscription to a person
    "ID",                        # unique ID of the subscription record itself
    "Contact - Contact type",    # used to remove staff, sales, press, accountants (not members)
    "Contact - Contact status",  # only used to remove Deceased contacts (never used to decide membership)
    "Status",                    # Active / Expired / Cancelled
    "Class",                     # Free / Full / Explorer  (free vs paid)
    "Grade",                     # Student / Recent Graduate / Professional
    "Duration",                  # 1 year or 3 years (needed so 3-year members are not counted as lapsed)
    "Subscription plan",         # Annual / 3 yr, Europe, standard / discounted (fee band)
    "Start date",                # when the subscription started
    "Renewal date",              # when it ends / is due for renewal
    "Cancelled at",              # when it was cancelled (only for cancelled records)
    "Sub status",                # 'Grade changed' marks a replacement record, not a real renewal
    "Current price",             # confirms free vs paid
    "Date created",              # when the record was created (migration records are dated June 2023)
]
# Dropped on purpose: 32 columns, e.g. Pause from/to, Renewal status, Default discount code (all empty)

subs = pd.read_excel(data_dir / "Subscriptions Europe.xlsx", usecols=SUBS_KEEP)    # load only the listed columns
subs = subs.rename(columns={                                                       # rename selected columns
    "Contact - ID": "contact_id", "ID": "subscription_id", "Contact - Contact type": "contact_type",
    "Contact - Contact status": "contact_status", "Status": "status", "Class": "member_class", "Grade": "grade",
    "Duration": "duration", "Subscription plan": "plan", "Start date": "start_date", "Renewal date": "renewal_date",
    "Cancelled at": "cancelled_at", "Sub status": "sub_status", "Current price": "price", "Date created": "date_created"})
subs["contact_id"] = subs["contact_id"].astype("string").str.strip().str.upper()   # tidy the ID text so it matches other files
print("Subscriptions Europe:", subs.shape)
for col in ["status", "member_class", "grade", "plan", "duration", "contact_type", "contact_status"]:
    print("\n", col); print(subs[col].value_counts(dropna=False))                  # look at the categories (check the 6 Grade values)

Subscriptions Europe: (16234, 15)

 status
status
Expired      11401
Active        3756
Cancelled     1015
Lapsed          57
Pending          5
Name: count, dtype: int64

 member_class
member_class
Free        12859
Full         2940
Explorer      435
Name: count, dtype: int64

 grade
grade
Professional Membership       10026
Student Membership             4274
Recent Graduate Membership     1885
Senior Active Membership         36
Staff                            12
Emeritus                          1
Name: count, dtype: int64

 plan
plan
Annual Europe, standard      15239
3 yr Europe, standard          512
Annual Europe, discounted      475
3 yr Europe, discounted          8
Name: count, dtype: int64

 duration
duration
1 year     15714
3 years      520
Name: count, dtype: int64

 contact_type
contact_type
Contact       13940
NaN            2238
Staff            22
Sales            16
Non Member       12
Accountant        6
Name: count, dtype: int64

 contact_status
contact_status
A

### 2.2 Individual Contacts (who the people are)

In [3]:
CONTACT_KEEP = [
    "Contact ID",                       # KEY
    "Default address - Country",        # geography (joins to the Country list)
    "Graduation date",                  # career stage (but 1 June 2018 is a migration default, so it is flagged)
    "Sector",                           # academia / industry / government
    "Retired or currently not working", # needed for the 'Retired' group
    "Opted in to be a mentor",          # an engagement signal
    "Topics interested in",             # turned into a COUNT of topics (a preference, not an activity)
    "Date created",
]
# Dropped on purpose: Job title, Primary organisation, Pronouns (personal details we do not need);
# the ten 'Show ...' directory columns (67% blank); Loyalty badge (94% blank and two conflicting definitions);
# SETAC Id, Website user ID, Is web user (extra IDs we do not use); Last in sync date/job ID, Date last modified (housekeeping);
# Contact type/status (already in the subscriptions file); Allow email / Opt out / Suppression list (not needed for our counts).

contacts = pd.read_excel(data_dir / "Individual Contacts.xlsx", usecols=CONTACT_KEEP)
contacts = contacts.rename(columns={
    "Contact ID": "contact_id", "Default address - Country": "country", "Graduation date": "graduation_date",
    "Sector": "sector", "Retired or currently not working": "retired_flag", "Opted in to be a mentor": "mentor",
    "Topics interested in": "topics", "Date created": "contact_created"})
contacts["contact_id"] = contacts["contact_id"].astype("string").str.strip().str.upper()
assert contacts["contact_id"].is_unique, "Individual Contacts should have one row per person"   # safety check

contacts["graduation_date"] = pd.to_datetime(contacts["graduation_date"], errors="coerce")
contacts["grad_is_default"] = contacts["graduation_date"] == pd.Timestamp("2018-06-01")        # flag the migration default date
contacts["n_topics"] = contacts["topics"].fillna("").str.split(",").map(lambda x: len([t for t in x if t.strip()]))   # how many topics were chosen
contacts["sector"] = contacts["sector"].fillna("Unknown")          # 29% have no sector: keep them as 'Unknown' instead of losing them
contacts = contacts.drop(columns=["topics", "contact_created"])    # drop the raw topic text and a date we no longer need
print("Individual Contacts:", contacts.shape)

Individual Contacts: (28922, 8)


In [4]:
contacts.head()

,contact_id,country,graduation_date,sector,retired_flag,mentor,grad_is_default,n_topics
0,000750DD-0E8A-4276-B8B013E4B5AE4567,Portugal,2026-09-30,Academia,0.0,False,False,3
1,00094A1F-9539-44DF-9B22339350548FA4,United States,2024-12-13,Academia,0.0,False,False,1
2,00106011-A0C2-42D2-BC06CB1ED87CA8AA,NaN,2018-06-01,Unknown,NaN,False,True,1
3,0011B39E-D714-46EB-A6AB32518CA13EBC,Austria,2023-12-20,Academia,0.0,False,False,8
4,0011CD8C-2757-47BA-8D0A7A50C26EBD23,South Africa,2005-04-22,Academia,0.0,False,False,1


### 2.3 Country list (geography and fee band)

In [5]:
countries = pd.read_excel(data_dir / "Country list.xlsx", usecols=["Country", "Geo unit", "Income level"])   # drop the two ID/label columns
countries = countries.rename(columns={"Country": "country", "Geo unit": "geo_unit", "Income level": "fee_band"})
assert countries["country"].is_unique, "Country list should have one row per country"

# Build ONE table of person details by merging contacts + country info. Every master table can reuse it.
contact_info = contacts.merge(countries, on="country", how="left", validate="many_to_one")   # many people -> one country row
unmatched = contact_info[contact_info["country"].notna() & contact_info["geo_unit"].isna()]
print("People whose country name is not in the Country list:", len(unmatched))
print(unmatched["country"].value_counts().head(10))               # if this list is not empty, the spellings need fixing
print("People with no country at all:", int(contact_info["country"].isna().sum()))

People whose country name is not in the Country list: 0
Series([], Name: count, dtype: int64)
People with no country at all: 1995


In [6]:
contact_info.head()

,contact_id,country,graduation_date,sector,retired_flag,mentor,grad_is_default,n_topics,geo_unit,fee_band
0,000750DD-0E8A-4276-B8B013E4B5AE4567,Portugal,2026-09-30,Academia,0.0,False,False,3,SETAC Europe,Standard
1,00094A1F-9539-44DF-9B22339350548FA4,United States,2024-12-13,Academia,0.0,False,False,1,SETAC North America,Standard
2,00106011-A0C2-42D2-BC06CB1ED87CA8AA,NaN,2018-06-01,Unknown,NaN,False,True,1,NaN,NaN
3,0011B39E-D714-46EB-A6AB32518CA13EBC,Austria,2023-12-20,Academia,0.0,False,False,8,SETAC Europe,Standard
4,0011CD8C-2757-47BA-8D0A7A50C26EBD23,South Africa,2005-04-22,Academia,0.0,False,False,1,SETAC Africa,Discounted


In [7]:
# Keep contacts with no country at all (1995) change the empty cells to Unknown, so they are not lost in the merge.

contact_info["country"] = contact_info["country"].fillna("Unknown")

In [8]:
contact_info["geo_unit"] = (
contact_info["geo_unit"]
.fillna("Unknown")
)
contact_info["fee_band"] = (
contact_info["fee_band"]
.fillna("Unknown")
)

### 2.4 Subscriptions All (used ONLY to spot members who moved to another SETAC region)

In [9]:
subs_all = pd.read_excel(data_dir / "Subscriptions All.xlsx", usecols=["Contact - ID", "Status", "Subscription plan", "Start date"])
subs_all = subs_all.rename(columns={"Contact - ID": "contact_id", "Status": "status", "Subscription plan": "plan", "Start date": "start_date"})
subs_all["contact_id"] = subs_all["contact_id"].astype("string").str.strip().str.upper()
subs_all["start_date"] = pd.to_datetime(subs_all["start_date"], errors="coerce")
print(subs_all["plan"].value_counts())        # CHECK: Europe plans contain the word 'Europe'; other regions have their own names

plan
Annual Europe, standard             15236
Annual North America, standard      12250
Annual Latin America, discounted     3253
Annual Africa, discounted            2764
Annual Asia-Pacific, standard        2738
Annual Asia-Pacific, discounted      2273
3 yr North America, standard          565
3 yr Europe, standard                 512
Annual Europe, discounted             475
Annual North America, discounted      327
3 yr Asia-Pacific, standard           180
3 yr Latin America, discounted         97
3 yr Asia-Pacific, discounted          93
3 yr Africa, discounted                51
3 yr North America, discounted         12
3 yr Europe, discounted                 8
Name: count, dtype: int64


## Step 3: Work out when each subscription starts and ends
We do this for **all** rows first (including rows with no Contact ID) so that we can measure how many real subscriptions the blank-ID decision removes.

In [10]:
for col in ["start_date", "renewal_date", "cancelled_at", "date_created"]:
    subs[col] = pd.to_datetime(subs[col], errors="coerce")        # make sure these columns are real dates

subs["term_years"] = pd.to_numeric(subs["duration"].astype(str).str.extract(r"(\d+)")[0], errors="coerce")   # '1 year' -> 1, '3 years' -> 3
subs["term_end"] = subs["start_date"] + pd.to_timedelta(subs["term_years"] * 365.25, unit="D")              # start date + term length

# Does 'Renewal date' behave like the END of the paid term? Compare it with start + term.
diff_days = (subs["renewal_date"] - subs["term_end"]).dt.days
agree = (diff_days.abs() <= 31).mean()
print(f"Renewal date is within 31 days of start + term for {agree:.1%} of subscriptions")
subs["end_date"] = subs["renewal_date"]      # Renewal date IS the end of cover (SETAC renews inside the same record by moving this date forward)
bad_end = subs["end_date"].isna() | (subs["end_date"] <= subs["start_date"]) | (subs["end_date"] > pd.Timestamp("2035-01-01"))   # missing, impossible or absurd (e.g. 2045) end dates
subs.loc[bad_end, "end_date"] = subs.loc[bad_end, "term_end"]                     # fall back to start + term for those
print("End dates replaced by start + term:", int(bad_end.sum()))     # should be a small number

# Cancelled subscriptions only count until the date they were cancelled
cancelled = subs["status"].eq("Cancelled")
with_date = cancelled & subs["cancelled_at"].notna()
subs.loc[with_date, "end_date"] = subs.loc[with_date, ["end_date", "cancelled_at"]].min(axis=1)   # cover stops at cancellation
subs.loc[cancelled & ~with_date, "end_date"] = subs.loc[cancelled & ~with_date, "start_date"]      # cancelled with no date = never valid

# Migration flags (the anomaly log showed one default start date on about 32% of rows)
top_start = subs["start_date"].value_counts()
default_start, default_n = top_start.index[0], int(top_start.iloc[0])
share = default_n / subs["start_date"].notna().sum()
print(f"Most common start date: {default_start.date()} on {default_n} rows ({share:.1%})")
subs["start_is_default"] = (subs["start_date"] == default_start) if share >= 0.10 else False   # TRUE = real join date is unknown
subs["migrated_record"] = subs["date_created"] < MIGRATION_END                                   # TRUE = record came from the old system
print("Records created before the migration cut-off:", int(subs["migrated_record"].sum()))
print(subs["date_created"].dt.to_period("M").value_counts().sort_index().head(6))     # check: a big spike in June 2023 is expected

Renewal date is within 31 days of start + term for 80.6% of subscriptions
End dates replaced by start + term: 20
Most common start date: 2023-07-02 on 5122 rows (31.6%)
Records created before the migration cut-off: 6812
date_created
2023-06    6812
2023-07     130
2023-08     109
2023-09     114
2023-10     263
2023-11     897
Freq: M, Name: count, dtype: int64


In [11]:
subs.head()

,contact_id,subscription_id,contact_type,contact_status,status,member_class,cancelled_at,price,duration,plan,grade,sub_status,renewal_date,start_date,date_created,term_years,term_end,end_date,start_is_default,migrated_record
0,2E210E6A-B6D1-49AE-BFA59D5FBBE29F0F,29F628CB-280D-43F1-AA2913998BA07802,Contact,Active,Active,Full,NaT,105.00,3 years,"3 yr Europe, discounted",Professional Membership,NaN,2027-11-20 09:17:46,2024-11-20 09:17:46,2024-11-20 09:17:46,3,2027-11-21 03:17:46,2027-11-20 09:17:46,False,False
1,518AA151-4C62-443A-9C950B92B42AF625,2A9B34C4-E8B1-4F70-AFEE760092B64DE4,Contact,Active,Active,Full,NaT,26.25,3 years,"3 yr Europe, discounted",Student Membership,NaN,2027-04-10 18:55:01,2024-04-10 18:55:01,2024-04-10 15:40:38,3,2027-04-11 12:55:01,2027-04-10 18:55:01,False,False
2,9C6ACFDC-CD42-429A-93DD3E2E04360480,2C4097BF-73E4-485A-9E35F18569602D05,Contact,Active,Active,Full,NaT,105.00,3 years,"3 yr Europe, discounted",Professional Membership,NaN,2028-03-05 15:06:28,2025-03-05 15:06:28,2025-03-03 10:53:15,3,2028-03-05 09:06:28,2028-03-05 15:06:28,False,False
3,10145152-F479-48BF-824215082F40CCBD,55D55D05-64FB-4303-9730BDA4B4DF7BF1,Contact,Active,Active,Full,NaT,105.00,3 years,"3 yr Europe, discounted",Professional Membership,NaN,2027-11-19 10:05:35,2024-11-19 10:05:35,2024-11-19 10:05:35,3,2027-11-20 04:05:35,2027-11-19 10:05:35,False,False
4,1F12DF2E-D408-4EA1-8397E3172FD5C097,89D9E178-F268-4A12-86C115AD05D6A114,Contact,Active,Active,Full,NaT,26.25,3 years,"3 yr Europe, discounted",Student Membership,NaN,2029-05-04 16:16:06,2026-05-04 16:16:06,2026-05-04 15:31:39,3,2029-05-04 10:16:06,2029-05-04 16:16:06,False,False


## Step 4: The blank Contact ID decision and who counts as a member
**Decision (team): exclude subscription rows with no Contact ID from every person-level analysis**, because without an ID we cannot tell whose subscription it is, so we cannot link renewals or attach sector and country. We do **not** delete them silently: we count them, test whether they differ from the rest, and save them.

In [12]:
blank = subs["contact_id"].isna()                                  # rows with no Contact ID
print(f"Rows with no Contact ID: {int(blank.sum())} of {len(subs)} ({blank.mean():.1%})")

# Do the excluded rows look different from the kept rows? (If they do, our results could be biased.)
mix = subs.groupby([blank.rename("blank_id"), "member_class"]).size().unstack(fill_value=0)   # rows per class: blank-ID vs linked
print("\nClass mix (counts):"); print(mix)
print("\nClass mix (% within each group):"); print((mix.div(mix.sum(axis=1), axis=0) * 100).round(1))
print("\nStart year (counts), blank-ID rows vs linked rows:")
print(subs.groupby([blank.rename("blank_id"), subs["start_date"].dt.year]).size().unstack(0, fill_value=0))

subs[blank].to_csv(cleaned_dir / "excluded_subscriptions_blank_contact_id.csv", index=False)   # keep a copy of what we excluded

Rows with no Contact ID: 2238 of 16234 (13.8%)

Class mix (counts):
member_class  Explorer   Free  Full
blank_id                           
False              429  10697  2870
True                 6   2162    70

Class mix (% within each group):
member_class  Explorer  Free  Full
blank_id                          
False              3.1  76.4  20.5
True               0.3  96.6   3.1

Start year (counts), blank-ID rows vs linked rows:
blank_id    False  True 
start_date              
2002            2      0
2003            1      0
2005            2      0
2006            1      0
2007            1      0
2009            2      0
2010            1      0
2011            1      0
2012            3      0
2013            1      0
2016            1      0
2017            2      1
2018            3      0
2019            1      0
2020           15      0
2021           63      2
2022          312     25
2023         5898   2126
2024         3174     76
2025         2559      8
2026        

2,238 subscription records (13.8%) excluded due to missing Contact IDs.

These records are disproportionately Free memberships (96.6% compared with 76.4% among linked records).

Results relating to Free members should therefore be interpreted with caution as the analytical dataset slightly under-represents this group.

- Approximately 95% of missing contact_ids originated from 2023 subscription records, consistent with the June 2023 system migration described by SETAC.
 
This suggests the missing-ID issue is largely related to historical migration activity rather than ongoing data collection processes.

In [13]:
# Who is a member? Apply the exclusion rules one at a time, in order, and count each.
reason = np.select(
    [subs["contact_id"].isna(),                                                    # 1. no Contact ID
     subs["contact_type"].eq("Staff"),                                             # 2. SETAC staff (SETAC confirmed: exclude)
     subs["contact_type"].notna() & (subs["contact_type"] != "Contact"),           # 3. sales, press, accountants, non-members
     subs["contact_status"].eq("Merged"),                                          # 4. old duplicate IDs: SETAC merged these into another contact
     subs["contact_status"].eq("Deceased"),                                        # 5. deceased contacts
     subs["start_date"].isna(),                                                    # 6. no start date
     subs["end_date"] <= subs["start_date"]],                                      # 7. cancelled before it began / zero-length
    ["no_contact_id", "staff", "not_a_member_contact_type", "merged_duplicate_contact", "deceased", "no_start_date", "never_valid_or_zero_length"],
    default="KEPT")
subs["exclude_reason"] = reason
exclusions = subs["exclude_reason"].value_counts().rename_axis("reason").reset_index(name="rows")
print(exclusions)
assert exclusions["rows"].sum() == len(subs)                                       # every row is accounted for
exclusions.to_csv(cleaned_dir / "exclusion_counts_subscriptions.csv", index=False)

recs = subs[subs["exclude_reason"] == "KEPT"].copy()                               # the valid subscription records we analyse
print("\nValid subscription records:", len(recs), "| people:", recs["contact_id"].nunique())
print(recs["contact_status"].value_counts())      # 'Merged' contacts may be duplicate people, hence the count should be noted

                       reason   rows
0                        KEPT  13494
1               no_contact_id   2238
2    merged_duplicate_contact    409
3   not_a_member_contact_type     34
4  never_valid_or_zero_length     29
5                       staff     22
6                    deceased      8

Valid subscription records: 13494 | people: 11141
contact_status
Active      13491
Inactive        3
Name: count, dtype: int64


In [14]:
term_days = recs["term_years"] * 365.25                                                         # length of one term in days
recs["n_terms"] = ((recs["end_date"] - recs["start_date"]).dt.days / term_days).round().clip(lower=1).astype(int)   # how many terms this record covers
print("Terms per record:"); print(recs["n_terms"].value_counts().sort_index())                  # check: most records should be 1 or 2

recs = recs.loc[recs.index.repeat(recs["n_terms"])].copy()                                      # repeat each record once per term
recs["record_id"] = recs["subscription_id"].astype(str)                                         # remember the original record ID
recs["term_no"] = recs.groupby("record_id").cumcount() + 1                                      # 1, 2, 3 ... within the record
is_last = recs["term_no"] == recs["n_terms"]                                                    # the last term ends at the Renewal date
first_start, last_end = recs["start_date"], recs["end_date"]                                    # keep the record's real start and end
recs["start_date"] = first_start + pd.to_timedelta((recs["term_no"] - 1) * recs["term_years"] * 365.25, unit="D")   # each term starts where the last one ended
recs["end_date"] = (first_start + pd.to_timedelta(recs["term_no"] * recs["term_years"] * 365.25, unit="D")).where(~is_last, last_end)   # the last term ends at the Renewal date
recs["subscription_id"] = recs["record_id"] + "_t" + recs["term_no"].astype(str)                # a unique ID for every term
assert recs["subscription_id"].is_unique                                                        # safety check
print("Rows after unpacking:", len(recs))

Terms per record:
n_terms
1     11881
2      1082
3       298
4       199
5        13
6         5
7         2
8         2
9         4
13        1
15        1
16        1
20        2
21        1
22        1
23        1
Name: count, dtype: int64
Rows after unpacking: 16046


## Step 5: Theme 1 logic: renewal, lapse, reinstatement, moved
**Rule reminders:** an Expired subscription is not automatically lost (check whether another follows). A 3-year subscription is ONE record, so it produces ONE renewal decision at its end and cannot be counted as lapsed in years 2 and 3. A "Grade changed" record is a replacement, not a renewal.

In [15]:
recs = recs.sort_values(["contact_id", "start_date", "end_date"]).reset_index(drop=True)    # each person's subscriptions in date order

# For each person, what is the latest end date seen so far? (This lets us ignore records that only overlap earlier cover.)
recs["running_max_end"] = recs.groupby("contact_id")["end_date"].cummax()
recs["previous_max_end"] = recs.groupby("contact_id")["running_max_end"].shift()           # the latest end date BEFORE this record
recs["extends_cover"] = recs["previous_max_end"].isna() | (recs["end_date"] > recs["previous_max_end"])   # TRUE = this record adds new months of cover

# A 'spell' = an unbroken period of membership. A new spell starts after a gap longer than the grace period.
recs["new_spell"] = recs["previous_max_end"].isna() | (recs["start_date"] > recs["previous_max_end"] + pd.Timedelta(days=GRACE_DAYS))
recs["spell_no"] = recs.groupby("contact_id")["new_spell"].cumsum()                        # 1 = first spell, 2 = they came back, ...
recs["spell_start"] = recs.groupby(["contact_id", "spell_no"])["start_date"].transform("min")
recs["spell_first_migrated"] = recs.groupby(["contact_id", "spell_no"])["migrated_record"].transform("first")   # was the spell's first record a migrated one?
print("People:", recs["contact_id"].nunique(), "| spells:", recs.groupby(["contact_id", "spell_no"]).ngroups)

People: 11141 | spells: 12125


In [16]:
# One row per RENEWAL DECISION = the end of a record that extends cover (and is not a 'Grade changed' replacement)
x = recs[recs["extends_cover"]].copy()
x["next_start"] = x.groupby("contact_id")["start_date"].shift(-1)          # when this person's next period of cover starts
x["next_class"] = x.groupby("contact_id")["member_class"].shift(-1)        # what class they moved to (free -> full etc.)
x["next_grade"] = x.groupby("contact_id")["grade"].shift(-1)
opps = x[~((x["sub_status"] == "Grade changed") & (x["term_no"] == x["n_terms"]))].copy()   # only the LAST term of a grade-changed record ends because of the change
opps = opps.rename(columns={"end_date": "due_date"})                       # the date the member had to decide

gap = pd.Timedelta(days=GRACE_DAYS)
opps["days_to_next"] = (opps["next_start"] - opps["due_date"]).dt.days     # negative = renewed early
opps["renewed"] = opps["next_start"].notna() & (opps["next_start"] <= opps["due_date"] + gap)          # RENEWED: next cover starts within the grace window
opps["reinstated_later"] = (~opps["renewed"]) & opps["next_start"].notna()                              # REINSTATED: not renewed, but came back later
opps["state"] = np.select(
    [opps["due_date"] > CUTOFF,                                            # not due yet at the cut-off
     (~opps["renewed"]) & (opps["due_date"] + gap > CUTOFF)],              # expired recently: too early to call
    ["not_yet_due", "pending"], default="resolved")
# Explorer and 3-year products ending after Aug 2026 cannot be renewed as the same product: keep them out of headline rates
opps["legacy_product_ended"] = (opps["member_class"].eq("Explorer") | opps["term_years"].eq(3)) & (opps["due_date"] > LAST_RENEWABLE)
opps["renewal_number"] = opps.groupby(["contact_id", "spell_no"]).cumcount() + 1       # 1st, 2nd, 3rd renewal decision in this spell
opps["tenure_years"] = (opps["due_date"] - opps["spell_start"]).dt.days / 365.25        # how long they had been a member at this decision
opps["tenure_reliable"] = ~opps["spell_first_migrated"].astype(bool)                    # FALSE = join date comes from the migration, so tenure is unreliable
print(opps["state"].value_counts())

state
resolved       11272
not_yet_due     3837
pending           61
Name: count, dtype: int64


In [17]:
# Did a non-renewer MOVE to another SETAC region (so we must not call them lost)?
other = subs_all[~subs_all["plan"].str.contains("Europe", case=False, na=False)]        # subscriptions in other regions
other = other[other["status"].ne("Cancelled")][["contact_id", "plan", "start_date"]].dropna()
other = other.rename(columns={"plan": "other_plan", "start_date": "other_start"})

check = opps.loc[~opps["renewed"], ["subscription_id", "contact_id", "due_date"]].merge(other, on="contact_id", how="inner")   # candidates: non-renewers who also appear elsewhere
window = pd.Timedelta(days=TRANSFER_DAYS)
check = check[(check["other_start"] >= check["due_date"] - window) & (check["other_start"] <= check["due_date"] + window)]       # started within +/- 180 days
check = check.sort_values("other_start").drop_duplicates("subscription_id")              # keep one match per decision
moved = check[["subscription_id", "other_plan"]]
opps = opps.merge(moved, on="subscription_id", how="left", validate="one_to_one")        # attach (blank if they did not move)

opps["outcome"] = np.select(
    [opps["state"].ne("resolved"), opps["renewed"], opps["other_plan"].notna()],
        [opps["state"].to_numpy(dtype=object), "renewed", "moved_unit"], default="expired")   # 'expired' = did not renew within 1 month (SETAC's meaning)
opps["in_headline_rate"] = opps["outcome"].isin(["renewed", "expired", "moved_unit"]) & ~opps["legacy_product_ended"]
opps["first_migration_renewal"] = opps["migrated_record"] & (opps["spell_no"] == 1) & (opps["term_no"] == 1)   # TRUE = the first renewal after the June 2023 migration (mostly due 1 July 2024)
print(opps["outcome"].value_counts())
print("Counts (always shown with percentages):")
head = opps[opps["in_headline_rate"]]["outcome"].value_counts()
print(pd.DataFrame({"count": head, "percent": (100 * head / head.sum()).round(1)}))

outcome
expired        8294
not_yet_due    3837
renewed        2956
pending          61
moved_unit       22
Name: count, dtype: int64
Counts (always shown with percentages):
            count  percent
outcome                   
expired      8294     73.6
renewed      2954     26.2
moved_unit     22      0.2


## Step 6: Theme 1 master tables
**Table A** `theme1_renewal_opportunities`: one row per renewal decision (Analyses 3, 5 and 6).
**Table B** `theme1_year_end_members`: one row per member per year-end (Analyses 1 and 2).
**Table C** `theme1_member_cohorts`: one row per person (Analysis 4).

In [18]:
def add_person_details(df):
    """Attach sector, country, region, fee band and career stage. Many rows -> one person row, so no rows are added."""
    n_before = len(df)
    df = df.merge(contact_info, on="contact_id", how="left", validate="many_to_one")   # LEFT join: keeps every subscription row
    assert len(df) == n_before, "A merge changed the number of rows"                   # safety check
    df["grade_short"] = df["grade"].str.replace(" Membership", "", regex=False)         # 'Student Membership' -> 'Student'
    df["member_type"] = np.where(df["retired_flag"] == 1, "Retired", df["grade_short"]) # Retired comes from the Contacts file, not from Grade
    df["paid_or_free"] = np.where(df["member_class"] == "Free", "Free", "Paid")         # Full and Explorer are paid
    return df

# TABLE A: renewal opportunities
t1_opps = add_person_details(opps)
keep_a = ["subscription_id", "contact_id", "spell_no", "renewal_number", "member_class", "paid_or_free", "grade_short", "member_type",
          "plan", "term_years", "price", "start_date", "due_date", "spell_start", "tenure_years", "tenure_reliable",
          "next_class", "next_grade", "days_to_next", "renewed", "reinstated_later", "other_plan", "outcome", "state",
          "legacy_product_ended", "in_headline_rate", "first_migration_renewal", "country", "geo_unit", "fee_band", "sector", "retired_flag", "grad_is_default"]
t1_opps = t1_opps[keep_a]                                                              # keep only the columns the analysis needs
t1_opps.to_csv(cleaned_dir / "theme1_renewal_opportunities.csv", index=False)
print("Table A:", t1_opps.shape)

Table A: (15170, 33)


In [19]:
def active_on(df, date):
    """One row per person who has valid cover on `date`. If they hold two subscriptions, keep the highest class (Full > Explorer > Free)."""
    c = df[(df["start_date"] <= date) & (df["end_date"] > date)].copy()                # cover includes this date
    c["class_rank"] = c["member_class"].map({"Full": 3, "Explorer": 2, "Free": 1}).fillna(0)
    c = c.sort_values(["contact_id", "class_rank", "start_date"]).drop_duplicates("contact_id", keep="last")
    return c[["contact_id", "member_class", "grade", "plan", "term_years", "start_date", "end_date"]]

snapshots = {2023: pd.Timestamp("2023-12-31"), 2024: pd.Timestamp("2024-12-31"), 2025: pd.Timestamp("2025-12-31"), 2026: CUTOFF}
stock = []
for year, date in snapshots.items():
    s = active_on(recs, date)
    s["snapshot_year"], s["snapshot_date"] = year, date
    stock.append(s)
t1_stock = add_person_details(pd.concat(stock, ignore_index=True))                     # TABLE B
t1_stock = t1_stock[["contact_id", "snapshot_year", "snapshot_date", "member_class", "paid_or_free", "grade_short", "member_type",
                     "plan", "term_years", "country", "geo_unit", "fee_band", "sector", "retired_flag"]]
t1_stock.to_csv(cleaned_dir / "theme1_year_end_members.csv", index=False)

# How much does the blank-ID exclusion change the headline count? (compare with ALL valid subscription rows, linked or not)
valid_all = subs[(subs["start_date"].notna()) & (subs["end_date"] > subs["start_date"])]
compare = pd.DataFrame({
    "linked_members": t1_stock.groupby("snapshot_year").size(),
    "all_valid_subscription_rows": {y: int(((valid_all["start_date"] <= d) & (valid_all["end_date"] > d)).sum()) for y, d in snapshots.items()}})
compare["difference"] = compare["all_valid_subscription_rows"] - compare["linked_members"]
print("Year-end membership: linked people vs all subscription rows (report both, so nobody is surprised):")
print(compare)
print("\nMembers by year and paid/free (counts):"); print(t1_stock.groupby(["snapshot_year", "paid_or_free"]).size().unstack(fill_value=0))

Year-end membership: linked people vs all subscription rows (report both, so nobody is surprised):
      linked_members  all_valid_subscription_rows  difference
2023            5498                         7821        2323
2024            3808                         3966         158
2025            3320                         3381          61
2026            3735                         3752          17

Members by year and paid/free (counts):
paid_or_free   Free  Paid
snapshot_year            
2023           3897  1601
2024           2666  1142
2025           2304  1016
2026           2804   931


In [20]:
# TABLE C: one row per person, for cohort retention (Analysis 4)
first = recs.sort_values(["contact_id", "start_date"]).drop_duplicates("contact_id")                  # each person's FIRST subscription
members = first[["contact_id", "start_date", "member_class", "grade", "migrated_record", "start_is_default"]].rename(
    columns={"start_date": "first_start", "member_class": "first_class", "grade": "first_grade",
             "migrated_record": "first_migrated", "start_is_default": "first_start_is_default"})
# Cohort label: only people who first appear AFTER the migration have a real join date
members["cohort"] = np.where(members["first_migrated"], "Legacy (joined before Jul 2023, date unknown)", members["first_start"].dt.year.astype(str))

# Retained after 1, 2, 3 years = still has valid cover one anniversary (+ grace) after the first start
m = recs[["contact_id", "start_date", "end_date"]].merge(members[["contact_id", "first_start"]], on="contact_id")
for k in (1, 2, 3):
    check_date = m["first_start"] + pd.Timedelta(days=365 * k + GRACE_DAYS)
    covered = m.loc[(m["start_date"] <= check_date) & (m["end_date"] > check_date), "contact_id"].unique()
    members[f"retained_{k}yr"] = members["contact_id"].isin(covered)
    members[f"observable_{k}yr"] = (members["first_start"] + pd.Timedelta(days=365 * k + GRACE_DAYS)) <= CUTOFF   # FALSE = not enough time has passed
t1_members = add_person_details(members.rename(columns={"first_class": "member_class", "first_grade": "grade"}))
t1_members.to_csv(cleaned_dir / "theme1_member_cohorts.csv", index=False)
print("Table C:", t1_members.shape)
print(t1_members.groupby("cohort").size())     # counts per cohort (a 2021 cohort cannot exist: the data starts in mid-2023)

Table C: (11141, 25)
cohort
2016                                                1
2022                                                6
2023                                             1246
2024                                             2418
2025                                             1739
2026                                             1290
Legacy (joined before Jul 2023, date unknown)    4441
dtype: int64


In [21]:
d = subs.copy()                                                  # work on a copy so nothing changes
d["gap_days"] = (d["renewal_date"] - d["term_end"]).dt.days      # renewal date minus (start + term)
d["agrees"] = d["gap_days"].abs() <= 31                          # True when the two dates match
print(pd.crosstab(d["status"], d["agrees"]))                     # which statuses disagree?
print(d.loc[~d["agrees"], "gap_days"].describe())                # by how many days?
active = d[d["status"] == "Active"]                              # subscriptions SETAC marks as Active
print("Active records:", len(active))
print("Active but our end date is already past:", int((active["end_date"] <= CUTOFF).sum()))   # a large number confirms the problem
print("Active with a Renewal date after the cut-off:", int((active["renewal_date"] > CUTOFF).sum()))

agrees     False  True 
status                 
Active      1069   2687
Cancelled    340    675
Expired     1731   9670
Lapsed        13     44
Pending        0      5
count    3153.000000
mean      273.891215
std       742.820271
min     -1096.000000
25%      -174.000000
50%       364.000000
75%       365.000000
max      8581.000000
Name: gap_days, dtype: float64
Active records: 3756
Active but our end date is already past: 9
Active with a Renewal date after the cut-off: 3755


In [22]:
from pathlib import Path                         # Path: handles folder locations
import pandas as pd                              # pandas: writes the Excel file

# 1. Choose where the file goes. 

save_folder = Path("/Users/Bosola/Downloads/SETAC-Team-12/Fourth Rev - Data Edited/Cleaned")
save_folder.mkdir(parents=True, exist_ok=True)   # create the folder if it does not exist yet
excel_path = save_folder / "SETAC_Theme1_master_tables.xlsx"   # the name of the Excel file

# 2. List the tables to save. Each one becomes its own sheet. (Sheet names can be at most 31 characters.)
tables_to_save = {
    "A_renewal_opportunities": t1_opps,          # master table A
    "B_year_end_members": t1_stock,              # master table B
    "C_member_cohorts": t1_members,              # master table C
    "person_details": contact_info,              # merged contacts + country list
    "valid_subscriptions": recs,                 # cleaned subscriptions used in the analysis
    "exclusion_counts": exclusions,              # how many rows were excluded and why
    "year_end_comparison": compare.reset_index() # linked members vs all subscription rows (reset_index turns the year into a normal column)
}

# 3. Write every table to its own sheet
with pd.ExcelWriter(excel_path, engine="openpyxl") as writer:        # open a new Excel file
    for sheet_name, table in tables_to_save.items():                 # go through the tables one by one
        table.to_excel(writer, sheet_name=sheet_name[:31], index=False)   # write the table without the row numbers
        sheet = writer.sheets[sheet_name[:31]]                       # get the sheet we just wrote
        sheet.freeze_panes = "A2"                                    # keep the header row visible when scrolling
        for column_cells in sheet.columns:                           # tidy the column widths
            longest = max(len(str(cell.value)) if cell.value is not None else 0 for cell in column_cells[:200])   # longest text in the first 200 rows
            sheet.column_dimensions[column_cells[0].column_letter].width = min(longest + 2, 40)   # width, capped at 40

# 4. Check that it worked
print("Saved:", excel_path)
check = pd.ExcelFile(excel_path)
for sheet_name in check.sheet_names:
    print(sheet_name, check.parse(sheet_name).shape)                 # rows and columns in each sheet

Saved: /Users/Bosola/Downloads/SETAC-Team-12/Fourth Rev - Data Edited/Cleaned/SETAC_Theme1_master_tables.xlsx
A_renewal_opportunities (15170, 33)
B_year_end_members (16361, 14)
C_member_cohorts (11141, 25)
person_details (28922, 10)
valid_subscriptions (16046, 31)
exclusion_counts (7, 2)
year_end_comparison (4, 4)
